# 🎓 HumanWriter AI - Fine-Tuning LLaMA 3.1 8B

Este notebook fine-tunea LLaMA 3.1 8B con el corpus académico dominicano usando Unsloth + LoRA.

**Requisitos:**
- Google Colab Pro (recomendado para GPU A100)
- Dataset preparado en formato Alpaca
- 2-4 horas de tiempo de entrenamiento

**Estimado de costos:**
- Colab Pro: $9.99/mes
- GPU T4: ~2-3 horas
- GPU A100: ~1-2 horas

## 1. Setup Environment

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Model loaded successfully!")

## 3. Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("✅ LoRA configured!")

## 4. Upload Dataset

Upload your `finetune_dataset.jsonl` file using the file upload button ⬅️

In [ ]:
from google.colab import files
import json

print("📤 Upload your finetune_dataset.jsonl file...")
uploaded = files.upload()

# Load dataset
dataset_path = list(uploaded.keys())[0]
print(f"✅ Dataset uploaded: {dataset_path}")

# Preview dataset
with open(dataset_path, 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
    print("\n📋 Sample entry:")
    print(json.dumps(sample, indent=2, ensure_ascii=False))

## 5. Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")

# Alpaca prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset prepared: {len(dataset)} examples")

## 6. Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "epoch",
    ),
)

print("✅ Trainer configured!")

## 7. Train Model

⏰ This will take 2-4 hours depending on dataset size and GPU.

In [ ]:
import time

start_time = time.time()

print("🚀 Starting training...")
print("This may take 2-4 hours. Don't close this tab!")
print("")

trainer_stats = trainer.train()

end_time = time.time()
elapsed = (end_time - start_time) / 60

print(f"\n✅ Training complete! Time: {elapsed:.2f} minutes")
print(f"📊 Final loss: {trainer_stats.training_loss:.4f}")

## 8. Test Model

In [ ]:
FastLanguageModel.for_inference(model)

instruction = "Escribe un texto académico sobre ingeniería industrial en República Dominicana"
input_text = ""

inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction,
            input_text,
            "",
        )
    ],
    return_tensors="pt"
).to("cuda")

print("🤖 Generating text...\n")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    top_p=0.9,
)

result = tokenizer.batch_decode(outputs)[0]
print(result)

print("\n✅ Model test complete!")

## 9. Export Model

In [ ]:
model.save_pretrained_merged("humanwriter_model", tokenizer, save_method="merged_16bit")
print("✅ Model saved in 16-bit format")

# Also save in GGUF format for Ollama
model.save_pretrained_gguf("humanwriter_model", tokenizer, quantization_method="q4_k_m")
print("✅ Model saved in GGUF Q4_K_M format (optimized for CPU)")

## 10. Download Model

In [ ]:
from google.colab import files
import os

# Find the GGUF file
gguf_files = [f for f in os.listdir("humanwriter_model") if f.endswith(".gguf")]

if gguf_files:
    gguf_path = f"humanwriter_model/{gguf_files[0]}"
    print(f"📥 Downloading: {gguf_path}")
    print(f"File size: {os.path.getsize(gguf_path) / (1024**3):.2f} GB")
    print("This may take several minutes...")
    
    files.download(gguf_path)
    
    print("✅ Download complete!")
    print("")
    print("📝 Next steps:")
    print("1. Move the downloaded file to your project")
    print("2. Run: ./scripts/export-model.sh <path-to-gguf-file>")
    print("3. The model will be available in Ollama")
else:
    print("❌ No GGUF file found. Check export step.")

## 🎉 All Done!

Your fine-tuned model is ready. Follow the instructions above to integrate it into HumanWriter AI.

**For specialized models:**
- Repeat this process with filtered datasets (only engineering docs, only social science docs, etc.)
- Name models accordingly: `humanwriter-ingenieria`, `humanwriter-sociales`, etc.

**Tips:**
- More training data = better quality
- Monitor loss to avoid overfitting
- Test thoroughly before deploying
- Keep base model as fallback